In [1]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
GPDS_MODEL_PATH = r"W:\SRH study\Case Study 2\Offline Signature Verification\src\embeddings\GPDS150\resnet\gpds_resnet_siamese.keras"

CEDAR_ORG_DIR  = r"W:\SRH study\Case Study 2\Offline Signature Verification\Datasets\signatures\full_org"
CEDAR_FORG_DIR = r"W:\SRH study\Case Study 2\Offline Signature Verification\Datasets\signatures\full_forg"

BHSIG_B_PATH = r"W:\SRH study\Case Study 2\Offline Signature Verification\Datasets\BHSig260\BHSig260-Bengali"
BHSIG_H_PATH = r"W:\SRH study\Case Study 2\Offline Signature Verification\Datasets\BHSig260\BHSig260-Hindi"

GPDS_THRESHOLD = 0.7038304805755615

print("GPDS model exists:", os.path.exists(GPDS_MODEL_PATH))
print("CEDAR genuine exists:", os.path.exists(CEDAR_ORG_DIR))
print("CEDAR forgery exists:", os.path.exists(CEDAR_FORG_DIR))
print("BHSig Bengali exists:", os.path.exists(BHSIG_B_PATH))
print("BHSig Hindi exists:", os.path.exists(BHSIG_H_PATH))

GPDS model exists: True
CEDAR genuine exists: True
CEDAR forgery exists: True
BHSig Bengali exists: True
BHSig Hindi exists: True


In [3]:
class SquaredEuclidean(layers.Layer):
    def call(self, inputs):
        a, b = inputs
        return tf.reduce_sum(tf.square(a - b), axis=1, keepdims=True)

In [4]:
siamese_model = keras.models.load_model(
    GPDS_MODEL_PATH,
    custom_objects={"SquaredEuclidean": SquaredEuclidean},
    compile=False
)

print("GPDS ResNet Siamese model loaded.")
siamese_model.summary()


GPDS ResNet Siamese model loaded.


Model: "siamese_resnet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ img_a (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ img_b (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet_embedding    │ (None, 128)       │ 24,145,152 │ img_a[0][0],      │
│ (Functional)        │                   │            │ img_b[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ d_sq                │ (None, 1)         │          0 │ resnet_embedding… │
│ (SquaredEuclidean)  │                   │            │ resnet_embedding… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,145,152 (92.11 MB)

 Trainable params: 557,440 (2.13 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [5]:
IMG_SIZE = (224, 224)

def tf_load_preprocess_resnet(path):
    img_bytes = tf.io.read_file(path)

    # Handle both jpg and png automatically
    img = tf.io.decode_image(img_bytes, channels=1, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)

    img = tf.image.grayscale_to_rgb(img)
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.resnet.preprocess_input(img)

    return img

In [13]:
def load_preprocess_resnet_cv2(path):
    path = path.numpy().decode("utf-8")
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Failed to read: {path}")

    img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA)
    img = np.stack([img, img, img], axis=-1).astype(np.float32)

    # ResNet preprocessing
    img = tf.keras.applications.resnet.preprocess_input(img)
    return img

def tf_load_preprocess_resnet_cv2(path):
    img = tf.py_function(load_preprocess_resnet_cv2, [path], Tout=tf.float32)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img

def make_pair_dataset_resnet_cv2(pairs_df, batch_size=32):
    a = pairs_df["path_a"].astype(str).values
    b = pairs_df["path_b"].astype(str).values
    y = pairs_df["label"].astype(np.float32).values

    ds = tf.data.Dataset.from_tensor_slices((a, b, y))

    def map_fn(pa, pb, lab):
        ia = tf_load_preprocess_resnet_cv2(pa)
        ib = tf_load_preprocess_resnet_cv2(pb)
        return (ia, ib), lab

    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

In [6]:
def make_pair_dataset_resnet(pairs_df, batch_size=32):
    a = pairs_df["path_a"].astype(str).values
    b = pairs_df["path_b"].astype(str).values
    y = pairs_df["label"].astype(np.float32).values

    ds = tf.data.Dataset.from_tensor_slices((a, b, y))

    def map_fn(pa, pb, lab):
        ia = tf_load_preprocess_resnet(pa)
        ib = tf_load_preprocess_resnet(pb)
        return (ia, ib), lab

    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

In [7]:
def get_distances(model, ds):
    d_all, y_all = [], []
    for (x, y) in ds:
        d_sq = model.predict(x, verbose=0).reshape(-1)
        d = np.sqrt(np.maximum(d_sq, 0.0))
        d_all.append(d)
        y_all.append(y.numpy().reshape(-1))
    return np.concatenate(d_all), np.concatenate(y_all)

def compute_far_frr(d, y, thresholds):
    y = y.astype(int)
    neg = (y == 0)
    pos = (y == 1)

    far, frr = [], []
    for t in thresholds:
        far.append(np.mean(d[neg] <= t))
        frr.append(np.mean(d[pos] > t))
    return np.array(far), np.array(frr)

def evaluate_cross_dataset(model, ds, transfer_threshold, dataset_name):
    d, y = get_distances(model, ds)

    thresholds = np.linspace(d.min(), d.max(), 800)
    far, frr = compute_far_frr(d, y, thresholds)

    eer_idx = np.argmin(np.abs(far - frr))
    eer = (far[eer_idx] + frr[eer_idx]) / 2
    best_t = thresholds[eer_idx]

    neg = (y == 0)
    pos = (y == 1)

    transfer_far = np.mean(d[neg] <= transfer_threshold)
    transfer_frr = np.mean(d[pos] > transfer_threshold)
    transfer_avg = (transfer_far + transfer_frr) / 2

    print(f"\n===== {dataset_name} =====")
    print("Distance min/mean/max:", d.min(), d.mean(), d.max())
    print("Cross-dataset EER:", float(eer))
    print("Best target threshold:", float(best_t))
    print("FAR@EER:", float(far[eer_idx]), "FRR@EER:", float(frr[eer_idx]))
    print(f"Using GPDS threshold ({transfer_threshold}):")
    print("Transfer FAR:", float(transfer_far))
    print("Transfer FRR:", float(transfer_frr))
    print("Transfer avg error:", float(transfer_avg))

    return {
        "dataset": dataset_name,
        "dist_min": float(d.min()),
        "dist_mean": float(d.mean()),
        "dist_max": float(d.max()),
        "eer": float(eer),
        "best_threshold": float(best_t),
        "far_at_eer": float(far[eer_idx]),
        "frr_at_eer": float(frr[eer_idx]),
        "transfer_threshold": float(transfer_threshold),
        "transfer_far": float(transfer_far),
        "transfer_frr": float(transfer_frr),
        "transfer_avg_error": float(transfer_avg),
    }

Cross validation on CEDAR

In [8]:
PAT_ORG  = re.compile(r"^original_(\d+)_(\d+)\.png$", re.IGNORECASE)
PAT_FORG = re.compile(r"^forgeries_(\d+)_(\d+)\.png$", re.IGNORECASE)

def build_cedar_df(org_dir, forg_dir):
    rows = []

    for fname in os.listdir(org_dir):
        if fname.lower().endswith(".png"):
            m = PAT_ORG.match(fname)
            if m:
                rows.append({
                    "writer_id": int(m.group(1)),
                    "sample_id": int(m.group(2)),
                    "label": "genuine",
                    "path": os.path.join(org_dir, fname)
                })

    for fname in os.listdir(forg_dir):
        if fname.lower().endswith(".png"):
            m = PAT_FORG.match(fname)
            if m:
                rows.append({
                    "writer_id": int(m.group(1)),
                    "sample_id": int(m.group(2)),
                    "label": "forgery",
                    "path": os.path.join(forg_dir, fname)
                })

    return pd.DataFrame(rows)

cedar_df = build_cedar_df(CEDAR_ORG_DIR, CEDAR_FORG_DIR)

print("CEDAR total images:", len(cedar_df))
print(cedar_df["label"].value_counts())
print("CEDAR writers:", cedar_df["writer_id"].nunique())
display(cedar_df.head())

CEDAR total images: 2640
label
genuine    1320
forgery    1320
Name: count, dtype: int64
CEDAR writers: 55


,writer_id,sample_id,label,path
0,10,1,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...
1,10,10,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...
2,10,11,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...
3,10,12,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...
4,10,13,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...


In [9]:
def build_pools(df_subset):
    genuine_by_writer = {}
    forgery_by_writer = {}

    for wid, g in df_subset.groupby("writer_id"):
        gg = g[g["label"] == "genuine"]["path"].tolist()
        ff = g[g["label"] == "forgery"]["path"].tolist()

        if len(gg) >= 2:
            genuine_by_writer[wid] = gg
        if len(ff) >= 1:
            forgery_by_writer[wid] = ff

    return genuine_by_writer, forgery_by_writer

def generate_pairs(df, writer_set, n_pairs=20000, seed=42, neg_mix=0.9):
    df_subset = df[df["writer_id"].isin(writer_set)].copy()
    genuine_by_writer, forgery_by_writer = build_pools(df_subset)

    writers = sorted(genuine_by_writer.keys())
    writers_with_forg = sorted(set(genuine_by_writer) & set(forgery_by_writer))

    rng = np.random.default_rng(seed)
    n_pos = n_pairs // 2
    n_neg = n_pairs - n_pos
    n_neg_same = int(round(n_neg * neg_mix))
    n_neg_cross = n_neg - n_neg_same

    pairs = []

    for _ in range(n_pos):
        w = rng.choice(writers)
        g = genuine_by_writer[w]
        i, j = rng.choice(len(g), size=2, replace=False)
        pairs.append({"path_a": g[i], "path_b": g[j], "label": 1})

    for _ in range(n_neg_same):
        w = rng.choice(writers_with_forg)
        g = genuine_by_writer[w]
        f = forgery_by_writer[w]
        i = rng.integers(0, len(g))
        j = rng.integers(0, len(f))
        pairs.append({"path_a": g[i], "path_b": f[j], "label": 0})

    for _ in range(n_neg_cross):
        w1, w2 = rng.choice(writers, size=2, replace=False)
        g1 = genuine_by_writer[w1]
        g2 = genuine_by_writer[w2]
        i = rng.integers(0, len(g1))
        j = rng.integers(0, len(g2))
        pairs.append({"path_a": g1[i], "path_b": g2[j], "label": 0})

    return pd.DataFrame(pairs).sample(frac=1.0, random_state=seed).reset_index(drop=True)

In [10]:
cedar_writers = set(cedar_df["writer_id"].unique())
cedar_pairs = generate_pairs(cedar_df, cedar_writers, n_pairs=20000, seed=42, neg_mix=0.9)

cedar_ds = make_pair_dataset_resnet(cedar_pairs, batch_size=32)

cedar_results = evaluate_cross_dataset(
    siamese_model,
    cedar_ds,
    GPDS_THRESHOLD,
    "GPDS model → CEDAR"
)


===== GPDS model → CEDAR =====
Distance min/mean/max: 0.09196752 0.7486836 1.8875382
Cross-dataset EER: 0.33504999999999996
Best target threshold: 0.6762583255767822
FAR@EER: 0.3357 FRR@EER: 0.3344
Using GPDS threshold (0.7038304805755615):
Transfer FAR: 0.3624
Transfer FRR: 0.3084
Transfer avg error: 0.33540000000000003


Cross validation on BHSig

In [11]:
def load_bhsig(root_path, script_type):
    records = []

    for writer_folder in sorted(os.listdir(root_path)):
        writer_path = os.path.join(root_path, writer_folder)
        if not os.path.isdir(writer_path):
            continue

        for fname in os.listdir(writer_path):
            if not fname.lower().endswith(".tif"):
                continue

            parts = fname.split("-")
            if len(parts) < 5:
                continue

            writer_num = int(parts[2])
            gf = parts[3].upper()
            sample_id = int(parts[4].split(".")[0])

            label = "genuine" if gf == "G" else "forgery"

            records.append({
                "dataset": f"BHSIG_{script_type}",
                "writer_id": f"{script_type}_{writer_num:03d}",
                "sample_id": sample_id,
                "label": label,
                "path": os.path.join(writer_path, fname)
            })

    return pd.DataFrame(records)

bhsig_b = load_bhsig(BHSIG_B_PATH, "B")
bhsig_h = load_bhsig(BHSIG_H_PATH, "H")
bhsig_df = pd.concat([bhsig_b, bhsig_h], ignore_index=True)

print("BHSig total images:", len(bhsig_df))
print(bhsig_df["label"].value_counts())
print("BHSig writers:", bhsig_df["writer_id"].nunique())
display(bhsig_df.head())

BHSig total images: 14040
label
forgery    7800
genuine    6240
Name: count, dtype: int64
BHSig writers: 260


,dataset,writer_id,sample_id,label,path
0,BHSIG_B,B_001,1,forgery,W:\SRH study\Case Study 2\Offline Signature Ve...
1,BHSIG_B,B_001,2,forgery,W:\SRH study\Case Study 2\Offline Signature Ve...
2,BHSIG_B,B_001,3,forgery,W:\SRH study\Case Study 2\Offline Signature Ve...
3,BHSIG_B,B_001,4,forgery,W:\SRH study\Case Study 2\Offline Signature Ve...
4,BHSIG_B,B_001,5,forgery,W:\SRH study\Case Study 2\Offline Signature Ve...


In [14]:
bhsig_writers = set(bhsig_df["writer_id"].unique())
bhsig_pairs = generate_pairs(bhsig_df, bhsig_writers, n_pairs=20000, seed=42, neg_mix=0.9)

#bhsig_ds = make_pair_dataset_resnet(bhsig_pairs, batch_size=32)
bhsig_ds = make_pair_dataset_resnet_cv2(bhsig_pairs, batch_size=32)

bhsig_results = evaluate_cross_dataset(
    siamese_model,
    bhsig_ds,
    GPDS_THRESHOLD,
    "GPDS model → BHSig"
)


===== GPDS model → BHSig =====
Distance min/mean/max: 0.0 0.43757275 1.7243195
Cross-dataset EER: 0.35085
Best target threshold: 0.35608601570129395
FAR@EER: 0.3492 FRR@EER: 0.3525
Using GPDS threshold (0.7038304805755615):
Transfer FAR: 0.7218
Transfer FRR: 0.0827
Transfer avg error: 0.40225


In [15]:
results_df = pd.DataFrame([cedar_results, bhsig_results])
display(results_df)

,dataset,dist_min,dist_mean,dist_max,eer,best_threshold,far_at_eer,frr_at_eer,transfer_threshold,transfer_far,transfer_frr,transfer_avg_error
0,GPDS model → CEDAR,0.091968,0.748684,1.887538,0.33505,0.676258,0.3357,0.3344,0.70383,0.3624,0.3084,0.33540
1,GPDS model → BHSig,0.000000,0.437573,1.724319,0.35085,0.356086,0.3492,0.3525,0.70383,0.7218,0.0827,0.40225
